# Notebook de preprocesamiento de los datos

En esta Jupyter Notebook se incluyen los script usaods para limpiar los datos recogidos de ejecuciones normales e infectadas de ransomwares de las distintas MVs. Para esto se han seguido los siguientes pasos:

- **Se ha modificado los nombres de los archivos.** Como los datos se tenian que pasar de maquinas infectadas a una maquina de control, se han pasado fragmentados en pequeños archivos. Homogeneizar sus nombres es una fase necesaria para su posterior tratamiento.
- **Eliminacion de algunos archivos corruptos.** Por la naturaleza del malware testeado, cuando la fase de encriptacion alzanzaba algunos de los scripts en ejecucion o los logs que se generan, estos se encriptaban total o parcialmente antes del envío a la maquina de control. Los archivos que no han podido recuperarse, se han eliminado.
- **Solución de pequeña errata.** Debido a un error en el script que se encarga de pasar los datos de una MV a otra, al comienzo de todos los csv se encontraba la cadena "nombre del archivo |||". Este se ha eliminado de todos los archivos.
- **Agrupacion de los logs**. En vez de tener, por ejemplo, scripts del tipo eventos_0, eventos_1 etc... se han agrupado todos en un solo log llamado eventos. Los logs resultantes de este proceso son pues *events.csv*, *filesystem_event.csv*, *HW_resources.csv*, *packets.csv*, *processes.csv* y *services.csv*.
- **Eliminación de los archivos redundantes**. Una vez la información esta unificada, se eliminan todos los logs que componian la informacion unificada, quedando en cada directorio los archivos ya comentados y alguna nota informativa (el SHA256 del malware en caso de ser una muestra infectada, por ejemplo).

In [36]:
import os
import glob
import re
# Elimina el header incorrecto de los archivos CSV en el directorio actual y subdirectorios. También arregla un defecto 
# renombrando 'network_data0.csv' a 'packets_0.csv'.

def clean_header(dir = '.'):
    patron = os.path.join(dir, '**', '*.csv')
    archivos_csv = glob.glob(patron, recursive=True)
    for archivo in archivos_csv:
        try:
            with open(archivo, 'r', encoding='utf-8') as f:
                contenido = f.read()
            nombre_archivo = os.path.basename(archivo)
            patron_incorrecto = f'{nombre_archivo}|||' 
            src = os.path.dirname(archivo)
            
            if nombre_archivo == 'network_data0.csv':
                new_name = os.path.join(src, 'packets_0.csv')
                os.rename(archivo, new_name) 
                print(f"Renombrado: {archivo} -> {new_name}")
                
            if re.match(r'services_monitor[0-9]*', nombre_archivo):
                aux = nombre_archivo.split('monitor')[1]    
                numero = (aux.split('.')[0])
                name = f'services_{numero}.csv'
                new_name = os.path.join(src, name)
                os.rename(archivo, new_name)
                
            if re.match(r'process_monitor[0-9]*', nombre_archivo):
                aux = nombre_archivo.split('monitor')[1]    
                numero = (aux.split('.')[0])
                name = f'processes_{numero}.csv'
                new_name = os.path.join(src, name)
                os.rename(archivo, new_name)    
                
            if re.match(r"filesystem_event[0-9]*", nombre_archivo):
                aux = nombre_archivo.split('event')[1] 
                numero = (aux.split('.')[0])
                name = f'filesystem_event_{numero}.csv'
                new_name = os.path.join(src, name)
                os.rename(archivo, new_name)
                
            if re.match(r"events[0-9]*", nombre_archivo):
                aux = nombre_archivo.split('s')[1] 
                numero = (aux.split('.')[0])
                name = f'events_{numero}.csv'
                new_name = os.path.join(src, name)
                os.rename(archivo, new_name)
                
        except Exception as e:
            print(f'Error al procesar {archivo}: {e}')
            
clean_header()

In [37]:
import glob
import os
import re
import csv

def add_csv_to_csv(archivo_origen, archivo_destino):
    with open(archivo_origen, 'r', encoding='utf-8') as src_file:
        reader = csv.reader(src_file)
        rows = list(reader)
        
    with open(archivo_destino, 'a', newline='', encoding='utf-8') as dest_file:
        writer = csv.writer(dest_file)
        if os.path.getsize(archivo_destino) == 0:
            writer.writerow(rows[0])  # Write header if file is empty
        for row in rows[1:]:  # Skip header
            writer.writerow(row)
        

def put_all_logs_in_one(dir = '.'):
    
    patron = os.path.join(dir, '**', '*.csv')
    patron_events = re.compile(r"events_[0-9]*")
    patron_services = re.compile(r"services_[0-9]*")
    patron_processes = re.compile(r"processes_[0-9]*")
    patron_packets = re.compile(r"packets_[0-9]*")
    patron_hw = re.compile(r"HW_resources_[0-9]*")
    patron_directories = re.compile(r"filesystem_event_[0-9]*")
    archivos_csv = glob.glob(patron, recursive=True)
    
    for archivo in archivos_csv:
        path = os.path.dirname(archivo)
        nombre_archivo = os.path.basename(archivo)
        if (patron_events.match(nombre_archivo)):
            archivo_destino = os.path.join(path, 'events.csv')
            add_csv_to_csv(archivo, archivo_destino)
        elif (patron_services.match(nombre_archivo)):
            archivo_destino = os.path.join(path, 'services.csv')
            add_csv_to_csv(archivo, archivo_destino)
        elif (patron_processes.match(nombre_archivo)):
            archivo_destino = os.path.join(path, 'processes.csv')
            add_csv_to_csv(archivo, archivo_destino)
        elif (patron_packets.match(nombre_archivo)):
            archivo_destino = os.path.join(path, 'packets.csv')
            add_csv_to_csv(archivo, archivo_destino)
        elif (patron_hw.match(nombre_archivo)):
            archivo_destino = os.path.join(path, 'HW_resources.csv')
            add_csv_to_csv(archivo, archivo_destino)
        elif (patron_directories.match(nombre_archivo)):
            archivo_destino = os.path.join(path, 'filesystem_event.csv')
            add_csv_to_csv(archivo, archivo_destino)

put_all_logs_in_one()

In [38]:
import os

def delete_files(dir = '.'):
    patron = os.path.join(dir, '**', '*.csv')
    patron_events = re.compile(r"events_[0-9]*")
    patron_services = re.compile(r"services_[0-9]*")
    patron_processes = re.compile(r"processes_[0-9]*")
    patron_packets = re.compile(r"packets_[0-9]*")
    patron_hw = re.compile(r"HW_resources_[0-9]*")
    patron_directories = re.compile(r"filesystem_event_[0-9]*")
    archivos_csv = glob.glob(patron, recursive=True)
    
    for archivo in archivos_csv:
        path = os.path.dirname(archivo)
        nombre_archivo = os.path.basename(archivo)
        if (patron_events.match(nombre_archivo)):
            os.remove(archivo)
        elif (patron_services.match(nombre_archivo)):
            os.remove(archivo)
        elif (patron_processes.match(nombre_archivo)):
            os.remove(archivo)
        elif (patron_packets.match(nombre_archivo)):
            os.remove(archivo)
        elif (patron_hw.match(nombre_archivo)):
            os.remove(archivo)
        elif (patron_directories.match(nombre_archivo)):
            os.remove(archivo)
            
delete_files()